In [1]:
# Step 0. Setup environment
import pandas as pd
import numpy as np
import os

base_dir_raw = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata/csv_exports"
base_dir_concepts = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata/csv_concepts_exports"
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

os.makedirs(output_dir, exist_ok=True)

print("✅ Environment ready")

✅ Environment ready


In [2]:
# Step 1. Load source tables
ventilation = pd.read_csv(os.path.join(base_dir_concepts, "ventilation.csv"))
vent_settings = pd.read_csv(os.path.join(base_dir_concepts, "ventilator_setting.csv"))
icustays = pd.read_csv(os.path.join(base_dir_raw, "icu_icustays.csv"))

print("✅ Data loaded")
print("Ventilation columns:", ventilation.columns.tolist())
print("Ventilator_setting columns:", vent_settings.columns.tolist())
print("ICU stays columns:", icustays.columns.tolist())

✅ Data loaded
Ventilation columns: ['stay_id', 'starttime', 'endtime', 'ventilation_status']
Ventilator_setting columns: ['subject_id', 'stay_id', 'charttime', 'respiratory_rate_set', 'respiratory_rate_total', 'respiratory_rate_spontaneous', 'minute_volume', 'tidal_volume_set', 'tidal_volume_observed', 'tidal_volume_spontaneous', 'plateau_pressure', 'peep', 'fio2', 'flow_rate', 'ventilator_mode', 'ventilator_mode_hamilton', 'ventilator_type']
ICU stays columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


In [4]:
# Step 2. Process ventilation.csv (ventilation status intervals)
vent_intervals = ventilation.rename(columns={
    "starttime": "vent_start_time",
    "endtime": "vent_stop_time",
    "ventilation_status": "vent_cat"
})

# Join with icustays to get subject_id and hadm_id
vent_intervals = vent_intervals.merge(
    icustays[["stay_id", "subject_id", "hadm_id"]],
    on="stay_id",
    how="left"
)

# Rename hadm_id → csn for consistency
vent_intervals = vent_intervals.rename(columns={"hadm_id": "csn", "subject_id": "pat_id"})

# Make sure datetime parsing is correct
vent_intervals["vent_start_time"] = pd.to_datetime(vent_intervals["vent_start_time"], errors="coerce")
vent_intervals["vent_stop_time"] = pd.to_datetime(vent_intervals["vent_stop_time"], errors="coerce")

print("✅ Ventilation intervals processed:", vent_intervals.shape)
vent_intervals.head()

✅ Ventilation intervals processed: (144812, 6)


,stay_id,vent_start_time,vent_stop_time,vent_cat,pat_id,csn
0,30000153,2174-09-29 12:01:00,2174-09-29 20:00:00,InvasiveVent,12466550,23998182
1,30000153,2174-09-29 20:00:00,2174-10-01 00:57:00,SupplementalOxygen,12466550,23998182
2,30000213,2162-06-21 05:45:00,2162-06-21 17:35:00,InvasiveVent,13180007,27543152
3,30000484,2136-01-14 18:46:00,2136-01-17 04:00:00,SupplementalOxygen,18421337,22413411
4,30000646,2194-04-29 01:40:00,2194-05-03 16:00:00,SupplementalOxygen,12207593,22795209


In [5]:
# Step 3. Process ventilator_setting.csv (detailed measurements)
vent_settings_small = vent_settings.rename(columns={
    "charttime": "recorded_time",
    "ventilator_type": "vent_name"
})[[
    "subject_id", "stay_id", "recorded_time",
    "vent_name", "ventilator_mode", "ventilator_mode_hamilton",
    "respiratory_rate_set", "tidal_volume_set",
    "tidal_volume_observed", "tidal_volume_spontaneous",
    "peep", "fio2"
]]

vent_settings_small["recorded_time"] = pd.to_datetime(
    vent_settings_small["recorded_time"], errors="coerce"
)

# Merge ventilator_mode + ventilator_mode_hamilton into one column
vent_settings_small["vent_mode"] = vent_settings_small["ventilator_mode"].combine_first(
    vent_settings_small["ventilator_mode_hamilton"]
)

print("✅ Ventilator settings processed:", vent_settings_small.shape)
vent_settings_small.head()

✅ Ventilator settings processed: (1377514, 13)


,subject_id,stay_id,recorded_time,vent_name,ventilator_mode,ventilator_mode_hamilton,respiratory_rate_set,tidal_volume_set,tidal_volume_observed,tidal_volume_spontaneous,peep,fio2,vent_mode
0,10000690,37081114,2150-11-02 20:27:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN
1,10000690,37081114,2150-11-03 00:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN
2,10000690,37081114,2150-11-03 04:38:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
3,10000690,37081114,2150-11-03 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN
4,10000690,37081114,2150-11-03 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN


In [6]:
# Step 4. Merge intervals (vent_cat) into settings
# Merge on stay_id; then filter so recorded_time falls within interval
vent_merged = vent_settings_small.merge(
    vent_intervals[["stay_id", "pat_id", "csn", "vent_start_time", "vent_stop_time", "vent_cat"]],
    on="stay_id",
    how="left"
)

# Keep only rows where recorded_time is within the interval
mask = (
    (vent_merged["recorded_time"] >= vent_merged["vent_start_time"]) &
    (vent_merged["recorded_time"] <= vent_merged["vent_stop_time"])
)
vent_merged = vent_merged.loc[mask].copy()

print("✅ Ventilation intervals aligned with settings:", vent_merged.shape)
vent_merged.head()

✅ Ventilation intervals aligned with settings: (1144856, 18)


,subject_id,stay_id,recorded_time,vent_name,ventilator_mode,ventilator_mode_hamilton,respiratory_rate_set,tidal_volume_set,tidal_volume_observed,tidal_volume_spontaneous,peep,fio2,vent_mode,pat_id,csn,vent_start_time,vent_stop_time,vent_cat
0,10000690,37081114,2150-11-02 20:27:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,10000690.0,25860671.0,2150-11-02 19:40:00,2150-11-05 16:00:00,SupplementalOxygen
1,10000690,37081114,2150-11-03 00:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,10000690.0,25860671.0,2150-11-02 19:40:00,2150-11-05 16:00:00,SupplementalOxygen
2,10000690,37081114,2150-11-03 04:38:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,10000690.0,25860671.0,2150-11-02 19:40:00,2150-11-05 16:00:00,SupplementalOxygen
3,10000690,37081114,2150-11-03 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,10000690.0,25860671.0,2150-11-02 19:40:00,2150-11-05 16:00:00,SupplementalOxygen
4,10000690,37081114,2150-11-03 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,10000690.0,25860671.0,2150-11-02 19:40:00,2150-11-05 16:00:00,SupplementalOxygen


In [7]:
# Step 5. Add placeholder column vent_tidal_rate_exhaled
vent_merged["vent_tidal_rate_exhaled"] = "Not Yet Decided"

print("✅ Added placeholder column vent_tidal_rate_exhaled")
vent_merged[[
    "pat_id", "csn", "recorded_time",
    "tidal_volume_observed", "tidal_volume_spontaneous", "vent_tidal_rate_exhaled"
]].head()

✅ Added placeholder column vent_tidal_rate_exhaled


,pat_id,csn,recorded_time,tidal_volume_observed,tidal_volume_spontaneous,vent_tidal_rate_exhaled
0,10000690.0,25860671.0,2150-11-02 20:27:00,NaN,NaN,Not Yet Decided
1,10000690.0,25860671.0,2150-11-03 00:50:00,NaN,NaN,Not Yet Decided
2,10000690.0,25860671.0,2150-11-03 04:38:00,NaN,NaN,Not Yet Decided
3,10000690.0,25860671.0,2150-11-03 08:00:00,NaN,NaN,Not Yet Decided
4,10000690.0,25860671.0,2150-11-03 09:00:00,NaN,NaN,Not Yet Decided


In [8]:
# Step 6. Select and reorder final columns
desired_columns = [
    "csn", "pat_id", "respiratory_rate_set", "tidal_volume_set", "tidal_volume_observed", 
    "tidal_volume_spontaneous", "vent_tidal_rate_exhaled", "peep", "fio2", "recorded_time",
    "vent_start_time", "vent_stop_time", "vent_name", "vent_mode", "vent_cat"
]

vent_final = vent_merged[desired_columns].rename(columns={
    "respiratory_rate_set": "vent_rate_set",
    "tidal_volume_set": "vent_tidal_rate_set"
})

print("✅ Final VENT table ready:", vent_final.shape)
vent_final.head()

✅ Final VENT table ready: (1144856, 15)


,csn,pat_id,vent_rate_set,vent_tidal_rate_set,tidal_volume_observed,tidal_volume_spontaneous,vent_tidal_rate_exhaled,peep,fio2,recorded_time,vent_start_time,vent_stop_time,vent_name,vent_mode,vent_cat
0,25860671.0,10000690.0,NaN,NaN,NaN,NaN,Not Yet Decided,NaN,70.0,2150-11-02 20:27:00,2150-11-02 19:40:00,2150-11-05 16:00:00,NaN,NaN,SupplementalOxygen
1,25860671.0,10000690.0,NaN,NaN,NaN,NaN,Not Yet Decided,NaN,50.0,2150-11-03 00:50:00,2150-11-02 19:40:00,2150-11-05 16:00:00,NaN,NaN,SupplementalOxygen
2,25860671.0,10000690.0,NaN,NaN,NaN,NaN,Not Yet Decided,NaN,100.0,2150-11-03 04:38:00,2150-11-02 19:40:00,2150-11-05 16:00:00,NaN,NaN,SupplementalOxygen
3,25860671.0,10000690.0,NaN,NaN,NaN,NaN,Not Yet Decided,NaN,100.0,2150-11-03 08:00:00,2150-11-02 19:40:00,2150-11-05 16:00:00,NaN,NaN,SupplementalOxygen
4,25860671.0,10000690.0,NaN,NaN,NaN,NaN,Not Yet Decided,NaN,70.0,2150-11-03 09:00:00,2150-11-02 19:40:00,2150-11-05 16:00:00,NaN,NaN,SupplementalOxygen


In [9]:
# Step 7. Save flat file
out_path = os.path.join(output_dir, "VENT.csv")
vent_final.to_csv(out_path, index=False)

print(f"✅ VENT.csv saved to {out_path} with shape {vent_final.shape}")
print("Final columns:", vent_final.columns.tolist())

✅ VENT.csv saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/VENT.csv with shape (1144856, 15)
Final columns: ['csn', 'pat_id', 'vent_rate_set', 'vent_tidal_rate_set', 'tidal_volume_observed', 'tidal_volume_spontaneous', 'vent_tidal_rate_exhaled', 'peep', 'fio2', 'recorded_time', 'vent_start_time', 'vent_stop_time', 'vent_name', 'vent_mode', 'vent_cat']
